# Analysis

**Hypothesis**: Within each anonymized cardiac cell population, spatial proximity between cells is associated with more similar transcriptional states, and this spatial–transcriptional coupling strength differs systematically across populations and samples, reflecting distinct modes of tissue organization in the developing human heart.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Set up visualization defaults for better plots
sc.settings.verbosity = 3
sc.settings.figsize = (8, 8)
sc.settings.dpi = 100
sc.settings.facecolor = 'white'
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['savefig.dpi'] = 150
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)

# Load data
print("Loading data...")
adata = sc.read_h5ad("/home/mingqiam/TissueAgent/demo/data/dataset_farah_anon.h5ad")
print(f"Data loaded: {adata.shape[0]} cells and {adata.shape[1]} genes")


Loading data...


Data loaded: 228635 cells and 238 genes


# Analysis Plan

**Hypothesis**: Within each anonymized cardiac cell population, spatial proximity between cells is associated with more similar transcriptional states, and this spatial–transcriptional coupling strength differs systematically across populations and samples, reflecting distinct modes of tissue organization in the developing human heart.

## Steps:
- Inspect and summarize available metadata and QC metrics, and formally define and store lists of well-represented populations per sample (and across samples) that will be used in all downstream, within-(Population, Sample_ID) spatial–transcriptional coupling analyses.
- For each well-represented (Population, Sample_ID) subset, compute per-cell spatial–transcriptional coupling scores by comparing transcriptional distances to k nearest spatial neighbors versus k random same-subset cells, and aggregate these to per-(Population, Sample_ID) coupling metrics.
- Within each well-represented (Population, Sample_ID), statistically test whether spatial neighbors are significantly more transcriptionally similar than random neighbors using paired tests on per-cell distance pairs, and summarize effect sizes, p-values, and usable cell counts.
- Compare spatial–transcriptional coupling strength across populations and samples using a compact summary table of per-(Population, Sample_ID) coupling metrics, and test for differences between populations with non-parametric tests while accounting for sample structure where possible.
- Relate spatial–transcriptional coupling metrics to population- and sample-level QC attributes (e.g., median Purity and UMI Count per (Population, Sample_ID)) using Spearman correlations, to assess whether higher-quality or more homogeneous groups show stronger spatial organization.
- In a selected subset of populations with particularly strong or weak coupling, identify genes whose expression is most associated with local spatial neighborhoods by correlating per-cell expression with neighbor-mean expression within (Population, Sample_ID), reporting ranked gene lists per population.


## This code refines the initial metadata/QC inspection by adding per-population-per-sample QC summaries and by defining stricter, reusable criteria for well-represented populations, then persists these summaries and population lists in adata.uns for use in all downstream coupling analyses.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

# Step 1: Inspect metadata, summarize per-population statistics, and define well-represented populations

# Basic AnnData overview (printed as text only)
print("AnnData object shape:", adata.shape)
print("obs columns:", list(adata.obs.columns))
print("obsm keys:", list(adata.obsm.keys()))
print("uns keys:", list(adata.uns.keys()))

# Confirm key metadata fields exist
required_obs_cols = ["Populations", "Sample_ID", "Batch", "UMI Count", "Purity"]
missing_cols = [col for col in required_obs_cols if col not in adata.obs.columns]
if missing_cols:
    print("Missing required obs columns; downstream steps may fail:", missing_cols)
else:
    print("All required obs columns are present.")

# Check that an embedding and clustering are available
print("Has X_umap in obsm:", "X_umap" in adata.obsm.keys())
print("Has spatial in obsm:", "spatial" in adata.obsm.keys())
print("Has leiden clustering in obs:", "leiden" in adata.obs.columns)

# Per-population cell counts overall and by sample
pop_counts = adata.obs["Populations"].value_counts().sort_values(ascending=False)
print("\nPer-population cell counts (all samples):")
print(pop_counts)

pop_sample_counts = adata.obs.groupby(["Populations", "Sample_ID"]).size().unstack(fill_value=0)
print("\nPer-population-by-sample cell counts (first 30 rows):")
print(pop_sample_counts.head(30))

# Per-population QC summaries for UMI Count and Purity (across all samples)
qc_summary_pop = adata.obs.groupby("Populations").agg({
    "UMI Count": ["median", "mean", "std", "min", "max"],
    "Purity": ["median", "mean", "std", "min", "max"],
    "Sample_ID": "nunique"
})
print("\nPer-population QC summary (UMI Count, Purity, #samples):")
print(qc_summary_pop)

# Optional: per-population-per-sample QC summaries (useful for later QC–coupling analyses)
qc_summary_pop_sample = adata.obs.groupby(["Populations", "Sample_ID"]).agg({
    "UMI Count": ["median", "mean"],
    "Purity": ["median", "mean"],
    "Complexity": ["median", "mean"] if "Complexity" in adata.obs.columns else "median"
})
print("\nPer-population-per-sample QC summary (showing first 30 rows):")
print(qc_summary_pop_sample.head(30))

# Identify populations with sufficient cells per sample for downstream analyses
min_cells_per_sample = 200  # heuristic threshold; can be adjusted
min_samples_with_sufficient_cells = 2  # require at least this many samples meeting the threshold

sufficient_pops_any = []  # populations with >= threshold cells in at least one sample
sufficient_pops_multi = []  # populations with >= threshold cells in at least min_samples_with_sufficient_cells samples
sufficient_pops_per_sample = {}  # dict: (pop) -> list of samples with sufficient cells

for pop, row in pop_sample_counts.iterrows():
    sufficient_samples = list(row[row >= min_cells_per_sample].index)
    if len(sufficient_samples) > 0:
        sufficient_pops_any.append(pop)
        sufficient_pops_per_sample[pop] = sufficient_samples
    if len(sufficient_samples) >= min_samples_with_sufficient_cells:
        sufficient_pops_multi.append(pop)

print(f"\nPopulations with at least {min_cells_per_sample} cells in at least one sample:")
print(sufficient_pops_any)
print(f"\nPopulations with at least {min_cells_per_sample} cells in at least {min_samples_with_sufficient_cells} samples:")
print(sufficient_pops_multi)
print("\nPer-population list of samples with sufficient cells (threshold per sample):")
for pop, samples in sufficient_pops_per_sample.items():
    print(pop, ":", samples)

# Persist selection in adata.uns for downstream steps
if "analysis_metadata" not in adata.uns:
    adata.uns["analysis_metadata"] = {}

adata.uns["analysis_metadata"]["pop_sample_counts"] = pop_sample_counts
adata.uns["analysis_metadata"]["qc_summary_pop"] = qc_summary_pop
adata.uns["analysis_metadata"]["qc_summary_pop_sample"] = qc_summary_pop_sample
adata.uns["analysis_metadata"]["sufficient_pops_any"] = sufficient_pops_any
adata.uns["analysis_metadata"]["sufficient_pops_multi"] = sufficient_pops_multi
adata.uns["analysis_metadata"]["sufficient_pops_per_sample"] = sufficient_pops_per_sample

print("\nStored population and QC summaries, as well as sufficient population lists, in adata.uns['analysis_metadata'].")

AnnData object shape: (228635, 238)
obs columns: ['Sample_ID', 'Batch', 'UMI Count', 'leiden', 'Complexity', 'Populations', 'Purity']
obsm keys: ['X_umap', 'spatial']
uns keys: ['dataset_provenance']
All required obs columns are present.
Has X_umap in obsm: True
Has spatial in obsm: True
Has leiden clustering in obs: True

Per-population cell counts (all samples):
Populations
PA     30380
PB     19947
PC     17584
PD     16624
PE     16511
PF     12248
PG     11596
PH     10887
PI     10441
PJ      9488
PK      8540
PL      8052
PM      7417
PN      7348
PO      5845
PP      5458
PQ      5429
PR      4673
PS      4599
PT      3726
PU      2356
PV      2292
PW      2027
PX      1562
PY      1292
PZ      1286
PAA     1027
Name: count, dtype: int64

Per-population-by-sample cell counts (first 30 rows):
Sample_ID    R77_4C4  R78_4C12  R78_4C15
Populations                             
PA              8718     10008     11654
PAA              413       392       222
PB              5945     

### Agent Interpretation

You’re in a very good position to probe the hypothesis with the current metadata/QC state:

1. **Power and design look strong for within-(Population, Sample_ID) tests.**  
   - Every population (PA–PZ) has ≥1,000 cells total, and most are several thousand.  
   - All but PV/PW are present with ≥200 cells in **all three** samples; even PV/PW/PX have ≥200 cells in ≥2 samples.  
   - This means you can:
     - Compute robust per-cell coupling metrics in essentially all populations.
     - Compare coupling across samples **within a population** (e.g., PA in R77_4C4 vs R78_4C12 vs R78_4C15).
     - Compare coupling across populations while partially accounting for sample effects (paired designs across the three samples).

   This directly supports the hypothesis that coupling strength differs systematically across populations/samples: your design allows many “triplicate” comparisons.

2. **QC structure is rich enough to test QC–coupling relationships later.**  
   - You have per-(Population, Sample_ID) summaries for:
     - UMI Count (median/mean)  
     - Purity (median/mean)  
     - Complexity (median/mean)  
   - There is substantial variation across populations in UMI median (e.g., ~150 in PG vs ~580 in PB vs ~545 in PL) and Purity (e.g., ~0.39 in PJ vs ~0.72 in PB; PS, PG, PR also relatively high).  
   - This heterogeneity is exactly what you need for step 5: checking if coupling is stronger in high-purity / high-depth / more complex groups.

3. **Some populations are especially promising to watch as “anchors” or contrasts.**  
   Based purely on QC patterns (not biology), these are likely to behave differently in spatial–transcriptional coupling:

   - **High-purity, moderate-to-high UMI groups (expect potentially strong coupling):**  
     - PB, PI, PS, PG, PR stand out for higher Purity means or medians, with reasonable UMI depth.  
     - PB: high UMI and high Purity across all three samples – good candidate for a “high-quality, possibly highly structured” population.  
     - PS/PG/PR: also higher Purity; if their spatial neighborhoods are well defined, they might show strong within-population spatial structure.

   - **Lower-purity or lower-depth groups (possible weak or noisy coupling):**  
     - PJ, PL, PK, PY have lower Purity medians/means; PG and PR are counterexamples (higher Purity with relatively low UMI), giving you variation in both directions.  
     - PG and PR have relatively low UMI medians but high Purity: they will be useful for disentangling the effects of UMI depth vs purity on coupling.

   - **Populations with very high counts across samples (high power):**  
     - PA–PE are all ~16k–30k cells each, present in all samples, so they’ll give very tight estimates of coupling.  
     - They also show clear shifts in median UMI between samples (e.g., in PA and many others, UMI medians increase from R77_4C4 → R78_4C12 → R78_4C15), giving leverage to see if sample-level technical conditions modulate coupling.

4. **Between-sample QC trends can inform later modeling.**  
   - Many populations show consistent **within-population shifts across samples**:
     - Example: PA median UMI: 334 (R77_4C4) → 404 (R78_4C12) → 499 (R78_4C15).  
     - Similar monotonic increases appear in PB, PC, PD, PE, etc.  
   - Purity and Complexity also vary by sample in a population-specific way.  
   - For later steps:
     - When comparing coupling across populations, it will be important to compare **within sample** (e.g., PB vs PG within R77_4C4) rather than only aggregating across samples, or to include Sample_ID as a blocking factor/random effect in non-parametric tests.
     - For step 5, you can compute coupling per (Population, Sample_ID) and correlate with sample-specific QC medians rather than global-population values, to reduce confounding.

5. **Code and data structures are well set up for the next analytical steps.**  
   - You’ve persisted:
     - `pop_sample_counts`  
     - `qc_summary_pop`  
     - `qc_summary_pop_sample`  
     - `sufficient_pops_*` lists  
   - This enables:  
     - Iteration over `sufficient_pops_per_sample[pop]` when computing per-(pop, sample) spatial–transcriptional coupling (step 2).  
     - Easy downstream joins between coupling metrics and QC summaries (`qc_summary_pop_sample`) in step 5.  
   - One technical suggestion for future code: ensure that any distance computations use a consistent expression space (e.g., log-normalized counts, maybe PCA on the 238 genes) and that you never pool across samples when defining neighbors; all steps should stay within (Population, Sample_ID) as planned.

6. **Concrete suggestions for the immediate next steps (2–3) given these results:**

   a. **Per-cell coupling computation (step 2):**
   - For each `pop` in `sufficient_pops_any` and each `sample` in `sufficient_pops_per_sample[pop]`:
     - Subset `adata_sub = adata[(adata.obs["Populations"]==pop) & (adata.obs["Sample_ID"]==sample)]`.
     - Build a spatial kNN graph on `adata_sub.obsm["spatial"]`. For robustness start with k ~ 5–10; you can later vary k and see if results are stable.
     - Define transcriptomic distance in a reduced space:
       - Normalize/log1p if not already, run PCA on the 238-gene expression, and use Euclidean or cosine distances on PCs.
     - For each cell:
       - Compute mean transcriptomic distance to its k spatial nearest neighbors.
       - Draw multiple sets of k **random same-(pop,sample)** cells (without spatial constraints) to estimate a null distribution of mean transcriptomic distance; store the per-cell observed-minus-expected or Z-score as your coupling score.
   - Aggregate:
     - Per-(pop, sample) mean, median, and distribution summaries of per-cell coupling scores.

   b. **Per-(pop, sample) statistical tests (step 3):**
   - Within each (pop, sample), perform a paired test:
     - For each cell, have: `d_spatial` vs `d_random_mean` (or set of random distances).  
     - Use Wilcoxon signed-rank test (non-parametric) on `d_spatial - d_random_mean`.
   - Output for each (pop, sample):
     - Effect size: e.g., median difference or Cohen’s d (or standardized Z from per-cell scores).  
     - P-value (with multiple-testing correction across all (pop, sample) combinations).  
     - Number of cells used.

   This will directly tell you, for each population and sample, whether spatial neighbors are significantly more transcriptionally similar than random neighbors, and how strong that effect is.

7. **How these results bear on the hypothesis so far:**
   - You’ve confirmed:
     - Spatial coordinates and clustering annotations exist.  
     - You have high cell counts across many anonymized populations and samples, and rich QC metrics.  
   - This supports the **feasibility** of the hypothesis test; you have enough within-(pop, sample) power to detect spatial–transcriptional coupling if it exists.
   - Actual **validation or refutation** of the hypothesis now hinges on:
     - Whether coupling metrics are consistently positive and significant in most populations, and  
     - Whether their magnitudes differ across populations/samples in systematic ways that correlate with QC and potentially with inferred biological identity later.

8. **Forward-looking ideas to make the analysis distinct and informative:**
   - Once you have coupling metrics:
     - Compare not just mean/median coupling but also the **heterogeneity** (variance) of coupling within a population: some populations may have highly structured “cores” and more random peripheries.  
     - Explore whether populations with **steeper UMI or Purity gradients across samples** also show larger shifts in coupling, which could hint at changes in tissue organization or technical confounding.  
     - Identify “outlier” populations where coupling is strong in one sample but weak in another; these will be candidates for deeper spatial pattern exploration and for the gene–neighborhood association analysis in step 6.

In summary, the current step confirms that the dataset and QC structure are well suited to test spatial–transcriptional coupling within and across anonymized populations and samples. The next priority is to implement the per-cell and per-(pop, sample) coupling metrics and tests, paying close attention to within-sample analyses and to the QC covariates you have already summarized.

## Next Steps
Step 1: Reuse the existing per-cell coupling metrics (e.g., d_spatial_mean, d_random_mean, delta, z_score) stored in adata.obs and the per-(Populations, Sample_ID) summary table in adata.uns['analysis_metadata']['coupling_summary'] as the sole basis for all downstream spatial–transcriptional coupling analyses, explicitly designating delta and z_score as primary coupling scores and fixing their sign conventions.
Step 2: Within each well-represented (Populations, Sample_ID) group defined in adata.uns['analysis_metadata']['sufficient_pops_per_sample'], formally test whether spatial neighbors are more transcriptionally similar than random neighbors using paired, non-parametric tests on per-cell (d_spatial_mean, d_random_mean) pairs, and append p-values and effect sizes to the existing coupling_summary table.
Step 3: Compare coupling strength across populations and samples by aggregating the updated coupling_summary: compute rank-based differences in delta_median and z_score_median between populations within each sample and across samples within each population, and report which populations consistently exhibit the strongest or weakest coupling.
Step 4: Assess how coupling strength and within-(Populations, Sample_ID) heterogeneity (e.g., variance or IQR of per-cell delta or z_score) relate to QC attributes such as median UMI Count, median Purity, and median Complexity using Spearman correlations across (Populations, Sample_ID) groups, leveraging the existing qc_summary_pop_sample metadata.
Step 5: In a focused set of populations selected from extremes of coupling effect size (e.g., top and bottom quartiles of delta_median or z_score_median), characterize the distribution of per-cell coupling scores and their association with local spatial density (e.g., mean spatial neighbor distance from cKDTree outputs or reused kNN distances) to distinguish uniformly structured versus spatially patchy populations.

## This code formally declares which existing per-cell coupling metrics (delta and z_score) will serve as the primary and secondary measures of spatial–transcriptional coupling, checks that the necessary per-cell and per-group results from the previous step are present, and stores explicit sign and interpretation conventions in adata.uns['analysis_metadata'] for use in all downstream steps.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# -----------------------------
# Configuration and checks
# -----------------------------

# Primary coupling metrics to use downstream
primary_coupling_col = "delta"   # positive => spatial neighbors more similar
secondary_coupling_col = "z_score"  # standardized version, also positive => stronger coupling

# Ensure required metadata from previous steps exists
if "analysis_metadata" not in adata.uns or "sufficient_pops_per_sample" not in adata.uns["analysis_metadata"]:
    raise ValueError("Expected 'analysis_metadata' with 'sufficient_pops_per_sample' from prior QC step.")

analysis_md = adata.uns.setdefault("analysis_metadata", {})

# Check that per-cell coupling metrics and summary table are present
required_obs_cols = ["d_spatial_mean", "d_random_mean", "delta", "z_score"]
missing_cols = [c for c in required_obs_cols if c not in adata.obs.columns]
if missing_cols:
    raise ValueError(
        f"Missing required per-cell coupling columns in adata.obs: {missing_cols}. "
        "Run the distance/coupling computation step before this cell."
    )

if "coupling_summary" not in analysis_md:
    raise ValueError(
        "Missing 'coupling_summary' in adata.uns['analysis_metadata']. "
        "Run the distance/coupling computation step before this cell."
    )

coupling_summary_df = analysis_md["coupling_summary"].copy()

# Sanity check sign conventions
summary_sign_check = coupling_summary_df[["delta_median", "z_score_median"]].describe()
print("Summary of coupling effect-size distributions (delta_median and z_score_median):")
print(summary_sign_check)

# Record which columns are designated as primary/secondary coupling metrics
analysis_md["coupling_metric_conventions"] = {
    "primary_coupling_col": primary_coupling_col,
    "secondary_coupling_col": secondary_coupling_col,
    "definition": {
        "delta": "mean(random transcriptional distance) - mean(spatial neighbor distance); positive => spatial neighbors more similar than random",
        "z_score": "standardized version of (random_mean - spatial_mean) so that larger positive values indicate stronger coupling",
    },
}

adata.uns["analysis_metadata"] = analysis_md

print("Using existing per-cell coupling metrics in adata.obs and per-(Populations, Sample_ID) summaries in adata.uns['analysis_metadata']['coupling_summary'] for all downstream analyses.")
print("Primary coupling metric:", primary_coupling_col)
print("Secondary coupling metric:", secondary_coupling_col)

ValueError: Missing required per-cell coupling columns in adata.obs: ['d_spatial_mean', 'd_random_mean', 'delta', 'z_score']. Run the distance/coupling computation step before this cell.

### Agent Interpretation

Current analysis step failed to run. Try an alternative approach

## Next Steps
Step 1: Compute fresh per-cell spatial–transcriptional coupling metrics within each sufficiently represented (Populations, Sample_ID) group by building a transcriptomic PCA space on log1p-normalized counts, defining k spatial neighbors per cell from obsm['spatial'], estimating null distances via repeated random same-group k-neighbor draws, and storing d_spatial_mean, d_random_mean, delta, and z_score in adata.obs while aggregating a new per-(Populations, Sample_ID) coupling_summary in adata.uns['analysis_metadata'].
Step 2: Within each well-represented (Populations, Sample_ID) group, formally test whether spatial neighbors are more transcriptionally similar than random neighbors using paired, non-parametric tests on per-cell (d_spatial_mean, d_random_mean) pairs, recording p-values and effect sizes in the coupling_summary table.
Step 3: Compare coupling strength and heterogeneity across populations and samples using the updated coupling_summary by computing rank-based contrasts in delta_median and z_score_median between populations within each sample and across samples within each population, and identify populations that consistently show strong, weak, or highly heterogeneous coupling.
Step 4: Relate spatial–transcriptional coupling strength and within-group heterogeneity (e.g., variance or IQR of per-cell delta or z_score) to QC attributes such as median UMI Count, median Purity, and median Complexity using Spearman correlations across (Populations, Sample_ID) groups, leveraging qc_summary_pop_sample stored in adata.uns['analysis_metadata'].

## This code recomputes per-cell spatial–transcriptional coupling metrics within each sufficiently represented (Populations, Sample_ID) group using PCA distances on log1p-normalized counts, with spatial k-nearest neighbors versus repeated random same-group neighbors to define delta and z_score, and then aggregates these into a per-group coupling_summary table. It also fixes RNG seeding, removes unused PCA KD-tree logic, and records explicit preprocessing and coupling metric conventions in adata.uns for consistent downstream analyses.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats
from scipy.spatial import cKDTree

# -----------------------------
# Step 1 (revised): de novo computation of per-cell spatial–transcriptional coupling
# -----------------------------

# Retrieve metadata from previous QC step
if 'analysis_metadata' not in adata.uns or 'sufficient_pops_per_sample' not in adata.uns['analysis_metadata']:
    raise ValueError("Expected 'analysis_metadata' with 'sufficient_pops_per_sample' from the QC step.")

analysis_md = adata.uns['analysis_metadata']

sufficient_pops_per_sample = analysis_md['sufficient_pops_per_sample']

# Check spatial coordinates
if 'spatial' not in adata.obsm:
    raise ValueError("Missing 'spatial' coordinates in adata.obsm; cannot compute spatial neighbors.")

# Ensure we have a transcriptomic representation; here we explicitly normalize/log1p and compute PCA
# on a copy to avoid altering adata.X in-place, and we record the preprocessing used.
if 'X_pca' not in adata.obsm_keys():
    tmp = adata.copy()
    # Assume raw counts and apply a standard normalization + log1p pipeline for coupling distances
    sc.pp.normalize_total(tmp, target_sum=1e4)
    sc.pp.log1p(tmp)
    sc.pp.pca(tmp, n_comps=min(30, tmp.n_vars))
    adata.obsm['X_pca'] = tmp.obsm['X_pca']
    # Record that coupling PCA was computed on normalize_total+log1p-transformed counts
    if 'analysis_metadata' not in adata.uns:
        adata.uns['analysis_metadata'] = {}
    adata.uns['analysis_metadata']['coupling_preproc'] = {
        'normalized': True,
        'log1p': True,
        'pca_n_comps': int(adata.obsm['X_pca'].shape[1]),
        'note': 'X_pca used for coupling distances was computed from normalize_total(target_sum=1e4) + log1p on a temporary copy.'
    }
    del tmp

X_pca = adata.obsm['X_pca']
X_spatial = adata.obsm['spatial']

# Parameters for coupling
k_spatial = 8  # number of spatial neighbors per cell
n_random_sets = 20  # number of random k-neighbor draws for null

n_cells = adata.n_obs

# Initialize per-cell coupling arrays (will remain NaN for cells not in sufficiently represented groups)
d_spatial_mean = np.full(n_cells, np.nan, dtype=float)
d_random_mean = np.full(n_cells, np.nan, dtype=float)

delta = np.full(n_cells, np.nan, dtype=float)
z_score = np.full(n_cells, np.nan, dtype=float)

# We will collect per-(Population, Sample_ID) summaries as we go
summary_records = []

# Map from global cell indices to (Populations, Sample_ID) for convenience
pops = adata.obs['Populations'].values
samples = adata.obs['Sample_ID'].values

# Instantiate RNG once for reproducible but non-identical randomization across groups
rng = np.random.default_rng(seed=0)

# Iterate over populations and their sufficiently represented samples
for pop, sample_list in sufficient_pops_per_sample.items():
    for sample in sample_list:
        mask = (pops == pop) & (samples == sample)
        idx = np.where(mask)[0]
        n_group = idx.size
        if n_group < (k_spatial + 1):
            continue  # not enough cells for k neighbors

        # Spatial kNN within this group
        spatial_coords_group = X_spatial[idx, :]
        spatial_tree = cKDTree(spatial_coords_group)
        # query k+1 because the first neighbor is the cell itself
        dists_spatial, knn_spatial = spatial_tree.query(spatial_coords_group, k=k_spatial + 1)
        # remove self (index 0)
        dists_spatial = dists_spatial[:, 1:]
        knn_spatial = knn_spatial[:, 1:]

        # For each cell in this group, compute mean transcriptomic distance to spatial neighbors
        for i_local, cell_global_idx in enumerate(idx):
            neighbor_global_idx = idx[knn_spatial[i_local]]
            d_vec = np.linalg.norm(X_pca[neighbor_global_idx] - X_pca[cell_global_idx], axis=1)
            d_spatial_mean[cell_global_idx] = float(d_vec.mean())

        # For null distances: for each cell, repeatedly sample k random same-group cells (excluding itself)
        group_indices = idx.copy()
        for i_local, cell_global_idx in enumerate(idx):
            pool = group_indices[group_indices != cell_global_idx]
            if pool.size < k_spatial:
                continue
            random_means = []
            for _ in range(n_random_sets):
                rand_neighbors = rng.choice(pool, size=k_spatial, replace=False)
                d_vec_rand = np.linalg.norm(X_pca[rand_neighbors] - X_pca[cell_global_idx], axis=1)
                random_means.append(d_vec_rand.mean())
            random_means = np.array(random_means)
            d_random_mean[cell_global_idx] = float(random_means.mean())
            # coupling: random_mean - spatial_mean (positive => spatial neighbors closer)
            if np.isfinite(d_spatial_mean[cell_global_idx]) and np.isfinite(d_random_mean[cell_global_idx]):
                delta[cell_global_idx] = d_random_mean[cell_global_idx] - d_spatial_mean[cell_global_idx]
                # standardize using null distribution; this is the standardized difference between expected-null mean and observed spatial mean
                if random_means.std(ddof=1) > 0:
                    z_score[cell_global_idx] = (d_random_mean[cell_global_idx] - d_spatial_mean[cell_global_idx]) / random_means.std(ddof=1)

        # After filling per-cell values for this group, compute per-(pop, sample) summary
        group_delta = delta[idx]
        group_z = z_score[idx]
        group_d_spatial = d_spatial_mean[idx]
        group_d_random = d_random_mean[idx]

        valid_mask = np.isfinite(group_delta) & np.isfinite(group_z)
        n_valid = int(valid_mask.sum())
        if n_valid == 0:
            continue

        rec = {
            'Populations': pop,
            'Sample_ID': sample,
            'n_cells': n_group,  # total cells in group
            'n_valid_cells': n_valid,  # cells contributing to coupling stats
            'delta_median': float(np.nanmedian(group_delta)),
            'delta_mean': float(np.nanmean(group_delta)),
            'delta_iqr': float(np.nanpercentile(group_delta, 75) - np.nanpercentile(group_delta, 25)),
            'z_score_median': float(np.nanmedian(group_z)),
            'z_score_mean': float(np.nanmean(group_z)),
            'z_score_iqr': float(np.nanpercentile(group_z, 75) - np.nanpercentile(group_z, 25)),
            'd_spatial_mean_median': float(np.nanmedian(group_d_spatial)),
            'd_random_mean_median': float(np.nanmedian(group_d_random))
        }
        summary_records.append(rec)

# Store per-cell metrics back into adata.obs
adata.obs['d_spatial_mean'] = d_spatial_mean
adata.obs['d_random_mean'] = d_random_mean
adata.obs['delta'] = delta
adata.obs['z_score'] = z_score

print("Per-cell coupling metrics computed and stored in adata.obs: 'd_spatial_mean', 'd_random_mean', 'delta', 'z_score'.")

# Build and store per-(Populations, Sample_ID) coupling summary
if len(summary_records) == 0:
    raise RuntimeError("No valid (Populations, Sample_ID) groups produced coupling summaries; check thresholds and data.")

coupling_summary_df = pd.DataFrame(summary_records)

if 'analysis_metadata' not in adata.uns:
    adata.uns['analysis_metadata'] = {}

adata.uns['analysis_metadata']['coupling_summary'] = coupling_summary_df

# Record coupling metric conventions (including the specific z-score definition)
adata.uns['analysis_metadata']['coupling_metric_conventions'] = {
    'primary_coupling_col': 'delta',
    'secondary_coupling_col': 'z_score',
    'definition': {
        'delta': 'mean(random transcriptional distance) - mean(spatial neighbor distance); positive => spatial neighbors more similar than random',
        'z_score': 'standardized version of (random_mean - spatial_mean) using within-cell null SD of random_means; larger positive values indicate stronger coupling (difference between expected-null mean and observed spatial mean, in SD units)'
    }
}

print("Per-(Populations, Sample_ID) coupling summary stored in adata.uns['analysis_metadata']['coupling_summary'].")
print("Number of (Populations, Sample_ID) groups summarized:", coupling_summary_df.shape[0])
print("Head of coupling_summary_df:\n", coupling_summary_df.head())

normalizing counts per cell


    finished (0:00:02)


computing PCA


    with n_comps=30


    finished (0:00:06)


Per-cell coupling metrics computed and stored in adata.obs: 'd_spatial_mean', 'd_random_mean', 'delta', 'z_score'.
Per-(Populations, Sample_ID) coupling summary stored in adata.uns['analysis_metadata']['coupling_summary'].
Number of (Populations, Sample_ID) groups summarized: 78
Head of coupling_summary_df:
   Populations Sample_ID  n_cells  n_valid_cells  delta_median  delta_mean  \
0          PA   R77_4C4     8718           8718      0.859734    0.864188   
1          PA  R78_4C12    10008          10008      1.336383    1.345498   
2          PA  R78_4C15    11654          11654      1.274153    1.274808   
3         PAA   R77_4C4      413            413      1.755234    1.942721   
4         PAA  R78_4C12      392            392      1.460443    1.551720   

   delta_iqr  z_score_median  z_score_mean  z_score_iqr  \
0   1.339181        1.085441      1.087047     1.687869   
1   1.475970        1.579759      1.587455     1.741761   
2   1.461181        1.463545      1.476427     1.6

### Agent Interpretation

The coupling computation step looks sound and the initial results are promising for your hypothesis.

Key points from the output  
- You successfully computed per-cell metrics (`d_spatial_mean`, `d_random_mean`, `delta`, `z_score`) for essentially all cells in the “sufficient” groups (e.g., PA and PAA examples show `n_valid_cells == n_cells`), so downstream group-wise summaries will be well powered.
- For the first few (Populations, Sample_ID) groups:
  - Median `delta` is clearly positive (e.g., ~0.86–1.34 for PA, ~1.46–1.76 for PAA), and median spatial distance is smaller than median random distance in each case (e.g., PA R77_4C4: 15.85 vs 16.69).
  - Median `z_score` values are consistently > 1 (up to ~1.7–1.8 in this small subset), with relatively large IQRs (~1.7–2.3), implying (i) spatial neighbors are more similar transcriptomically than random neighbors, and (ii) there is meaningful cell-to-cell heterogeneity in coupling within each group.

Taken together, for at least these populations/samples, the hypothesis that spatially proximate cells are more similar than random same-group cells is empirically supported at this descriptive level.

Specific feedback and suggestions for the next steps

1. **Next planned step (formal paired tests) is well-justified and should be informative**
   - You now have per-cell pairs (`d_spatial_mean`, `d_random_mean`) for each (Populations, Sample_ID).
   - Given the strongly positive group medians and the large `n_valid_cells` (hundreds to thousands), a paired non-parametric test (e.g. Wilcoxon signed-rank) per group will almost certainly yield very small p-values for most groups, but you should still:
     - Store both **effect sizes** (e.g., median `delta`, maybe also median percent difference) and **p-values** in `coupling_summary`.
     - Consider also reporting **paired Cohen’s d** or rank-biserial correlation per group, since p-values will saturate with large n.

   - One practical detail: for the test, operate on the **raw per-cell deltas** or on the paired distances directly, but apply a small filter:
     - Exclude cells where either `d_spatial_mean` or `d_random_mean` is NaN (though by design, that should be rare, given `n_valid_cells == n_cells` in your examples).

2. **Leverage the heterogeneity metrics you already computed**
   - The existing `delta_iqr` and `z_score_iqr` are exactly what you’ll need for the “heterogeneity” component of the hypothesis.
   - Next steps that will build on these:
     - Plot **within-group distributions** (e.g., violin/boxplots) of `delta` or `z_score` for selected populations across samples to visualize how coupling heterogeneity differs.
     - Examine whether groups with higher median coupling also have higher heterogeneity, or whether some groups show strong, *uniform* coupling vs strong but *bimodal* or wide distributions.

3. **Prepare for cross-population and cross-sample comparisons (Step 3)**
   - With 78 groups already in `coupling_summary`, you’re well set up for:
     - **Per-sample contrasts across populations**: for a given `Sample_ID`, rank populations by `delta_median` and `z_score_median` to see which populations are consistently high- or low-coupling in that anatomical context.
     - **Per-population contrasts across samples**: for each `Populations` label, look at the spread of `delta_median` across `Sample_ID`s to see whether coupling is stable or sample-specific.
   - Make sure to:
     - Add identifiers that facilitate grouping, e.g. ensure `coupling_summary` has categorical dtypes for Populations and Sample_ID.
     - Consider **normalizing or standardizing** some coupling metrics across samples for comparison, if you suspect sample-level technical differences (though your within-group, within-sample construction already mitigates many global confounders).

4. **Be cautious about interpretation of absolute distance scale**
   - PCA distances are unitless and influenced by the normalization and number of PCs. You’ve documented the preprocessing, which is good.
   - For cross-population comparison, it’s fine to use `delta` and `z_score` because they are **within-group standardized**. Still, explicitly emphasize that you’re comparing *relative* coupling strength, not absolute transcriptional distances.

5. **Consider a couple of robustness checks before deeply interpreting biological patterns**
   - **Vary k_spatial** (e.g., 5, 8, 12) for a subset of groups:
     - If coupling patterns (rank order of populations by `delta_median`) are robust, that strengthens the conclusion that you are not overfitting to an arbitrary neighborhood size.
   - **Check sensitivity to PCA dimensionality**:
     - For one or two samples, recompute X_pca with e.g. 20 and 40 comps and see if group-level `delta_median` correlates strongly with your current estimates.

   These can be done selectively so as not to duplicate the full pipeline.

6. **Plan ahead for QC covariate analyses (Step 4)**
   - You will soon relate coupling metrics to QC summaries (median UMI, Purity, Complexity).
   - Given the strong coupling signals already visible, test:
     - Whether groups with **low UMI / low Purity** show systematically **lower `z_score_median`** or higher `z_score_iqr` (i.e., noisier coupling).
     - Use **Spearman correlations** across the 78 groups and consider partial correlations or stratified analyses if certain populations dominate the range of QC metrics.
   - This will help separate genuine biological heterogeneity in coupling from technical variation.

7. **Sanity checks on the current coupling_summary contents**
   - Before moving on, inspect:
     - Distribution of `delta_median` and `z_score_median` across all 78 groups (histograms).
     - Relationship between `delta_median` and `z_score_median` (scatter plot) to ensure they are consistent and that the z-scores scale roughly linearly with delta magnitudes.
   - Identify:
     - Any groups with **near-zero or negative `delta_median`** (if they exist) as candidates for unusual spatial organization (weak or inverse coupling).
     - Groups with very high `delta_iqr` / `z_score_iqr` as “highly heterogeneous” in coupling and hence interesting for later biological interpretation.

How this bears on the hypothesis so far  
- The preliminary group-level statistics (strongly positive `delta_median`, `z_score_median` > 1, with substantial IQR) **support the hypothesis** that spatially proximal cells within a labeled population and sample are, on average, more transcriptomically similar than random same-group cells.
- The non-trivial IQRs for delta and z_score indicate that **within-group heterogeneity** in coupling exists and is quantifiable, aligning with the second part of your hypothesis.

Immediate next actions I’d suggest  
1. Implement Step 2: within each (Populations, Sample_ID), run a Wilcoxon signed-rank test comparing `d_spatial_mean` vs `d_random_mean` per cell; store p-values and effect sizes in `coupling_summary`.  
2. Generate quick summary plots across all 78 groups to:
   - Confirm that most have `delta_median > 0`.
   - Identify outlier groups with weak or highly heterogeneous coupling for focused follow-up in Steps 3–4.

## Next Steps
Step 1: Within each (Populations, Sample_ID) group represented in the coupling_summary, formally test whether spatial neighbors are more transcriptionally similar than random neighbors using paired, non-parametric tests on per-cell d_spatial_mean and d_random_mean (with clear handling/flagging of low-n groups), and append effect sizes and multiple-testing–corrected p-values to adata.uns['analysis_metadata']['coupling_summary'].
Step 2: Using the augmented coupling_summary, compare coupling strength and heterogeneity across populations and samples by ranking and contrasting (Populations, Sample_ID) groups on delta_median, z_score_median, and their dispersion measures, summarizing which populations within each sample and which samples within each population consistently show strong, weak, or highly heterogeneous coupling.
Step 3: Relate spatial–transcriptional coupling strength and within-group heterogeneity (e.g., delta_median, z_score_median, and their IQRs) to QC attributes such as median UMI Count, median Purity, and median Complexity using Spearman correlations across (Populations, Sample_ID) groups, leveraging qc_summary_pop_sample in adata.uns['analysis_metadata'], and report correlation coefficients and p-values.

## This code performs paired Wilcoxon signed-rank tests within each (Populations, Sample_ID) group to assess whether spatial neighbors are more transcriptomically similar than random neighbors, derives a consistent per-cell delta from the paired distances, computes a rank-biserial effect size, applies Benjamini–Hochberg FDR correction, and merges these statistics back into the coupling_summary table.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Retrieve analysis metadata and ensure required structures are present
if 'analysis_metadata' not in adata.uns:
    raise ValueError("Missing 'analysis_metadata' in adata.uns; run the QC/coupling computation step first.")

analysis_md = adata.uns['analysis_metadata']

if 'coupling_summary' not in analysis_md:
    raise ValueError("Missing 'coupling_summary' in adata.uns['analysis_metadata']; run the coupling computation step first.")

coupling_summary_df = analysis_md['coupling_summary'].copy()

required_obs_cols = ['d_spatial_mean', 'd_random_mean']
missing_cols = [c for c in required_obs_cols if c not in adata.obs.columns]
if missing_cols:
    raise ValueError(f"Required per-cell columns missing from adata.obs: {missing_cols}; cannot run paired tests.")

# Prepare containers for test results
results = []

pops = adata.obs['Populations'].values
samples = adata.obs['Sample_ID'].values

# Iterate over unique (Populations, Sample_ID) combinations represented in coupling_summary
for _, row in coupling_summary_df[['Populations', 'Sample_ID']].drop_duplicates().iterrows():
    pop = row['Populations']
    sample = row['Sample_ID']

    mask = (pops == pop) & (samples == sample)
    idx = np.where(mask)[0]
    if idx.size == 0:
        continue

    d_spatial = adata.obs['d_spatial_mean'].values[idx].astype(float)
    d_random = adata.obs['d_random_mean'].values[idx].astype(float)

    # Define per-cell delta from the paired distances to ensure consistency
    delta_vals = d_random - d_spatial

    # Only keep cells with finite paired distances
    valid = np.isfinite(d_spatial) & np.isfinite(d_random)
    n_valid = int(valid.sum())

    if n_valid < 10:
        # Too few valid paired observations for a reliable test; record descriptive stats only
        results.append({
            'Populations': pop,
            'Sample_ID': sample,
            'n_paired_cells': n_valid,
            'wilcoxon_stat': np.nan,
            'wilcoxon_pvalue': np.nan,
            'delta_median_cells': float(np.nanmedian(delta_vals[valid])) if n_valid > 0 else np.nan,
            'delta_mean_cells': float(np.nanmean(delta_vals[valid])) if n_valid > 0 else np.nan,
            'effect_size_rank_biserial': np.nan
        })
        continue

    d_spatial_valid = d_spatial[valid]
    d_random_valid = d_random[valid]
    delta_valid = delta_vals[valid]

    # Wilcoxon signed-rank test on paired distances: H0 median difference == 0
    try:
        stat, pval = stats.wilcoxon(d_spatial_valid, d_random_valid, alternative='less')
        # alternative='less': d_spatial < d_random expected if spatial neighbors are more similar
    except ValueError:
        # e.g., if all differences are zero
        stat, pval = np.nan, np.nan

    # Rank-biserial correlation as effect size for paired Wilcoxon
    # r_rb = (W_plus - W_minus) / (W_plus + W_minus), computed from signed ranks of nonzero differences
    diffs = d_random_valid - d_spatial_valid
    n = diffs.size  # includes zeros; r_rb uses only nonzero diffs
    nonzero = diffs != 0
    if nonzero.sum() > 0:
        ranks = stats.rankdata(np.abs(diffs[nonzero]))
        W_plus = ranks[diffs[nonzero] > 0].sum()
        W_minus = ranks[diffs[nonzero] < 0].sum()
        denom = W_plus + W_minus
        if denom > 0:
            r_rb = (W_plus - W_minus) / denom
        else:
            r_rb = np.nan
    else:
        r_rb = np.nan

    results.append({
        'Populations': pop,
        'Sample_ID': sample,
        'n_paired_cells': n,
        'wilcoxon_stat': float(stat) if np.isfinite(stat) else np.nan,
        'wilcoxon_pvalue': float(pval) if np.isfinite(pval) else np.nan,
        'delta_median_cells': float(np.nanmedian(delta_valid)),
        'delta_mean_cells': float(np.nanmean(delta_valid)),
        'effect_size_rank_biserial': float(r_rb) if np.isfinite(r_rb) else np.nan
    })

results_df = pd.DataFrame(results)

# Multiple-testing correction (Benjamini-Hochberg FDR) for Wilcoxon p-values
pvals = results_df['wilcoxon_pvalue'].values.copy()
valid_p = np.isfinite(pvals)
if valid_p.sum() > 0:
    order = np.argsort(pvals[valid_p])
    ranked_p = pvals[valid_p][order]
    m = float(valid_p.sum())
    bh_fdr = np.empty_like(ranked_p)
    # Benjamini-Hochberg step-up
    prev = 1.0
    for i in range(int(m) - 1, -1, -1):
        bh_val = ranked_p[i] * m / (i + 1.0)
        if bh_val < prev:
            prev = bh_val
        bh_fdr[i] = prev
    # place back into full array
    fdr_full = np.full_like(pvals, np.nan, dtype=float)
    fdr_full_indices = np.where(valid_p)[0][order]
    fdr_full[fdr_full_indices] = bh_fdr
    results_df['wilcoxon_fdr'] = fdr_full
else:
    results_df['wilcoxon_fdr'] = np.nan

# Merge the test results back into the coupling_summary table on (Populations, Sample_ID)
merged = coupling_summary_df.merge(
    results_df,
    on=['Populations', 'Sample_ID'],
    how='left',
    suffixes=('', '_test')
)

# Overwrite coupling_summary in analysis_metadata with the augmented version
adata.uns['analysis_metadata']['coupling_summary'] = merged

# Print a compact textual summary of test results
valid_fdr = np.isfinite(results_df['wilcoxon_fdr'])
if valid_fdr.any():
    prop_sig = float(np.mean(results_df.loc[valid_fdr, 'wilcoxon_fdr'] < 0.05))
else:
    prop_sig = np.nan

print("Number of (Populations, Sample_ID) groups tested:", results_df.shape[0])
print("Proportion of groups with FDR < 0.05 (among groups with finite FDR):", prop_sig)
print("Top 10 groups by strongest positive rank-biserial effect size:")
print(results_df.sort_values('effect_size_rank_biserial', ascending=False).head(10)[[
    'Populations', 'Sample_ID', 'n_paired_cells', 'effect_size_rank_biserial', 'wilcoxon_pvalue', 'wilcoxon_fdr', 'delta_median_cells'
]])


Number of (Populations, Sample_ID) groups tested: 78
Proportion of groups with FDR < 0.05 (among groups with finite FDR): 1.0
Top 10 groups by strongest positive rank-biserial effect size:
   Populations Sample_ID  n_paired_cells  effect_size_rank_biserial  \
55          PR  R78_4C12            1019                   0.993415   
54          PR   R77_4C4            1008                   0.991725   
34          PK  R78_4C12            2979                   0.987396   
33          PK   R77_4C4            2435                   0.985671   
19          PF  R78_4C12            4449                   0.984218   
35          PK  R78_4C15            3126                   0.982909   
51          PQ   R77_4C4            1872                   0.980471   
39          PM   R77_4C4            3181                   0.979825   
56          PR  R78_4C15            2646                   0.979595   
43          PN  R78_4C12            2994                   0.976395   

    wilcoxon_pvalue   wilcoxo

### Agent Interpretation

These results strongly support the core part of your hypothesis — within-group spatial neighbors are much more transcriptomically similar than random neighbors — and they give you a good quantitative handle to investigate how that coupling varies across groups.

Key points and implications:

1. **Robust within-group spatial–transcriptional coupling**

   - You tested 78 (Populations, Sample_ID) groups, and *100%* of those with finite FDR have `wilcoxon_fdr < 0.05`. This means that, essentially everywhere you had enough data, `d_spatial_mean < d_random_mean` at the per-cell level.
   - The rank-biserial effect sizes for the strongest groups are extremely high (0.97–0.99+), with large `n_paired_cells` (often >1000, up to ~4500). This is not a marginal effect; it’s a very strong, pervasive signal of spatial–transcriptional coupling.
   - Median per-cell deltas (`d_random_mean - d_spatial_mean`) in the top groups are on the order of ~2–3, which is substantial given these are averaged distances.

   Taken together, these results validate the first part of your hypothesis: within each cell population–sample group, spatial neighbors are *on average* more transcriptomically similar than random same-population cells.

2. **Code/analysis sanity and potential refinements**

   - The test direction (`alternative='less'` with `d_spatial` vs. `d_random`) and effect size definition (`d_random - d_spatial`) are consistent: positive delta and large positive rank-biserial correspond to stronger-than-random local similarity.
   - You appropriately drop groups with <10 valid pairs from hypothesis testing and still record descriptive metrics, which will be useful downstream.
   - A minor caveat: some groups have such large n and large effect that `wilcoxon_pvalue` underflows to 0.0. Your BH implementation handles this, but just be aware downstream that “0” here means “numerically indistinguishable from 0,” not literally.

   Possible small improvements (optional, not required to proceed):
   - Store the *sign* of the median delta relative to zero explicitly (e.g., a boolean “delta_median_positive”). Almost all will be positive, but if any groups are negative or near zero they’d be immediately visible.
   - Consider adding a measure of within-group dispersion for the per-cell delta (e.g., `delta_iqr_cells`) at this same step; that will be directly relevant for the “heterogeneity” part of the hypothesis.

3. **How this informs the next planned steps**

   You now have, per (Populations, Sample_ID), at least:
   - `delta_median_cells`
   - `delta_mean_cells`
   - `effect_size_rank_biserial`
   - `wilcoxon_pvalue`, `wilcoxon_fdr`
   - `n_paired_cells`

   These are exactly what you need to move into the “between-group” part of your hypothesis:

   **a. Comparing coupling strength across groups (next plan item)**

   - Use `delta_median_cells` and `effect_size_rank_biserial` as your primary “coupling strength” metrics.
   - Rank groups within each sample and within each population:
     - Within each `Sample_ID`: which populations have consistently high vs. low `delta_median_cells` and `effect_size_rank_biserial`?
     - Within each `Populations` code: how does coupling vary across samples?
   - The groups you highlighted (e.g., PK, PF, PQ, PM, PR in R77_4C4 and R78_4C12/4C15) are very strongly coupled. A useful summary would be:
     - Boxplots/violin plots of per-cell delta for a few “strong” vs. “weak” groups.
     - A heatmap or clustered matrix of `delta_median_cells` with Populations on one axis and Sample_ID on the other to visually identify patterns (e.g., some populations strong in all samples vs. sample-specific).

   Recommendations for this step:
   - Compute simple dispersion metrics per group from the per-cell delta you already have:
     - `delta_iqr_cells = IQR(d_random - d_spatial)`
     - `delta_sd_cells = std(d_random - d_spatial)`
   - Append these dispersion metrics into `coupling_summary` alongside the summary you just wrote; that will operationalize “heterogeneity” and is still distinct from what the paper likely did.

   **b. Identifying heterogeneity vs. homogeneity of coupling**

   - Once dispersion metrics are added, classify groups into:
     - Strong + low-heterogeneity (high delta_median, low IQR) → robustly coupled everywhere.
     - Strong + high-heterogeneity (high delta_median, high IQR) → some cells strongly coupled, others weak.
     - Weak + low-heterogeneity (low delta_median, low IQR) → uniformly weak coupling.
     - Weak + high-heterogeneity (low delta_median, high IQR) → mixed behavior.
   - This classification can be done by e.g. median splits or quantiles over `delta_median_cells` and `delta_iqr_cells`.

4. **Preparing for QC-related correlations (third plan item)**

   To address the “systematically varies” part of the hypothesis and guard against QC confounding:

   - Ensure that `qc_summary_pop_sample` includes, for the same set of (Populations, Sample_ID) pairs:
     - `median_UMI`, `median_purity`, `median_complexity` (or equivalent).
   - Merge this QC table with the updated `coupling_summary` on `(Populations, Sample_ID)`.
   - For each coupling metric you care about (`delta_median_cells`, `effect_size_rank_biserial`, and ideally `delta_iqr_cells`):
     - Compute Spearman correlation vs. each QC metric across groups.
     - Report ρ, p-value, and the number of groups used for each correlation.
   - Interpretation:
     - If coupling strength is *strongly* correlated with QC metrics (e.g., low complexity → low delta_median), then apparent differences across populations/samples might be QC-driven.
     - If correlations are weak or absent, you can more confidently interpret differences as biologically meaningful.

5. **Biological and spatial follow-up suggestions**

   Once you’ve ranked groups and looked at heterogeneity:

   - For a few exemplar populations:
     - Choose one strongly coupled and one more weakly coupled population.
     - Visualize:
       - Spatial map of cells colored by per-cell delta (high vs. low coupling) within a sample.
       - Distribution of delta within those groups.
   - This will help relate coupling to anatomical localization (e.g., some populations might have uniformly strong coupling in specific regions vs. more diffuse, heterogeneous patterns in others), while staying distinct from the original paper’s specific claims.

Summary of what’s validated and what to do next:

- The dataset shows very strong, statistically robust evidence that spatial neighbors are more transcriptomically similar than random neighbors within nearly all (Populations, Sample_ID) groups tested — validating the “existence” of spatial–transcriptional coupling.
- The next critical steps to address your full hypothesis are:
  1. Quantify and compare coupling strength and heterogeneity across groups (`delta_median_cells`, effect sizes, and newly computed dispersion metrics).
  2. Systematically relate these group-level coupling metrics to QC attributes with Spearman correlations, to separate biological variation from technical effects.
  3. Use ranked comparisons and simple classifications (strong/weak × homogeneous/heterogeneous) to identify especially interesting populations and samples for deeper spatial visualization.

## Next Steps
Step 1: Compare coupling strength and heterogeneity across (Populations, Sample_ID) groups by augmenting the existing coupling_summary with per-group dispersion metrics derived directly from per-cell delta (e.g., delta_iqr_cells, delta_sd_cells, n_delta_cells), then ranking and textually summarizing populations within each sample and samples within each population on delta_median_cells, effect_size_rank_biserial, and these heterogeneity metrics to identify consistently strong, weak, and highly heterogeneous coupling groups.
Step 2: Relate group-level coupling strength and heterogeneity metrics from the augmented coupling_summary (delta_median_cells, effect_size_rank_biserial, delta_iqr_cells) to QC attributes in qc_summary_pop_sample (median UMI Count, median Purity, median Complexity) using a merged coupling_qc_merged table and Spearman correlations across (Populations, Sample_ID) groups, printing correlation coefficients, p-values, and concise interpretive text on whether QC covaries with coupling strength or heterogeneity.

## This code augments the existing coupling_summary with within-group heterogeneity metrics derived directly from per-cell delta (d_random_mean - d_spatial_mean), then ranks and classifies (Populations, Sample_ID) groups by coupling strength and heterogeneity while preparing a merged coupling–QC table for downstream correlation analysis.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Retrieve analysis metadata and required tables
if 'analysis_metadata' not in adata.uns:
    raise ValueError("Missing 'analysis_metadata' in adata.uns; run prior steps first.")

analysis_md = adata.uns['analysis_metadata']

if 'coupling_summary' not in analysis_md:
    raise ValueError("Missing 'coupling_summary' in adata.uns['analysis_metadata'].")
if 'qc_summary_pop_sample' not in analysis_md:
    raise ValueError("Missing 'qc_summary_pop_sample' in adata.uns['analysis_metadata'].")

coupling_summary = analysis_md['coupling_summary'].copy()
qc_pop_sample = analysis_md['qc_summary_pop_sample'].copy()

# Ensure obs has the per-cell distances needed to recompute delta consistently
for col in ['d_spatial_mean', 'd_random_mean']:
    if col not in adata.obs.columns:
        raise ValueError(f"Required per-cell column '{col}' missing from adata.obs.")

# ------------------------------------------------------------------
# 1. Compute within-group dispersion of per-cell delta (heterogeneity)
#    and attach to coupling_summary without overwriting existing cols
# ------------------------------------------------------------------

pops = adata.obs['Populations'].values
samples = adata.obs['Sample_ID'].values

# Per-cell delta defined consistently with prior tests: d_random_mean - d_spatial_mean
cell_delta = (adata.obs['d_random_mean'].astype(float).values -
              adata.obs['d_spatial_mean'].astype(float).values)

# Container for dispersion stats per (Populations, Sample_ID)
het_records = []

cs_groups = coupling_summary[['Populations', 'Sample_ID']].drop_duplicates()

# Optional: warn if there are (Populations, Sample_ID) in obs but not in coupling_summary
obs_groups = pd.DataFrame({'Populations': pops, 'Sample_ID': samples}).drop_duplicates()
missing_in_cs = pd.merge(obs_groups, cs_groups, on=['Populations', 'Sample_ID'], how='left', indicator=True)
missing_in_cs = missing_in_cs[missing_in_cs['_merge'] == 'left_only'][['Populations', 'Sample_ID']]
if not missing_in_cs.empty:
    print("Warning: Some (Populations, Sample_ID) combinations in adata.obs are missing from coupling_summary;\n"
          "heterogeneity will only be computed for groups present in coupling_summary.")

for (pop, sample) in cs_groups.itertuples(index=False):
    mask = (pops == pop) & (samples == sample)
    if not np.any(mask):
        print(f"Warning: (Populations={pop}, Sample_ID={sample}) present in coupling_summary but not in adata.obs; skipping.")
        het_records.append({
            'Populations': pop,
            'Sample_ID': sample,
            'delta_iqr_cells': np.nan,
            'delta_sd_cells': np.nan,
            'n_delta_cells': 0
        })
        continue
    group_delta = cell_delta[mask]
    valid = np.isfinite(group_delta)
    n_valid = int(valid.sum())
    if n_valid == 0:
        het_records.append({
            'Populations': pop,
            'Sample_ID': sample,
            'delta_iqr_cells': np.nan,
            'delta_sd_cells': np.nan,
            'n_delta_cells': 0
        })
        continue
    gd = group_delta[valid]
    iqr = np.nanpercentile(gd, 75) - np.nanpercentile(gd, 25)
    sd = float(np.nanstd(gd, ddof=1)) if n_valid > 1 else np.nan
    het_records.append({
        'Populations': pop,
        'Sample_ID': sample,
        'delta_iqr_cells': float(iqr),
        'delta_sd_cells': sd,
        'n_delta_cells': n_valid
    })

het_df = pd.DataFrame(het_records)

# Avoid silently overwriting existing heterogeneity columns
for col in ['delta_iqr_cells', 'delta_sd_cells', 'n_delta_cells']:
    if col in coupling_summary.columns:
        print(f"Warning: column '{col}' already exists in coupling_summary and will be overwritten with recomputed values.")

# Merge heterogeneity metrics into coupling_summary
coupling_summary = coupling_summary.drop(columns=[c for c in ['delta_iqr_cells', 'delta_sd_cells', 'n_delta_cells']
                                                 if c in coupling_summary.columns])

coupling_summary = coupling_summary.merge(
    het_df,
    on=['Populations', 'Sample_ID'],
    how='left'
)

# Document the meaning of the new heterogeneity metrics
analysis_md.setdefault('coupling_metric_conventions', {})['delta_heterogeneity_metrics'] = {
    'delta_iqr_cells': 'IQR of per-cell delta = d_random_mean - d_spatial_mean within each (Populations, Sample_ID); larger values indicate more heterogeneous coupling.',
    'delta_sd_cells': 'Sample standard deviation of per-cell delta within each (Populations, Sample_ID).',
    'n_delta_cells': 'Number of cells with finite per-cell delta contributing to within-group heterogeneity metrics.'
}

# ------------------------------------------------------------------
# 2. Rank and summarize coupling strength/heterogeneity across groups
# ------------------------------------------------------------------

# Check for key strength metrics and enforce presence of delta_median_cells
for col in ['delta_median_cells', 'effect_size_rank_biserial', 'wilcoxon_fdr']:
    if col not in coupling_summary.columns:
        msg = f"Expected column '{col}' not found in coupling_summary."
        if col == 'delta_median_cells':
            raise ValueError(msg + " This metric is required for strength ranking; ensure the Wilcoxon step was run.")
        else:
            print("Warning:", msg, "Downstream summaries for this metric will be limited.")

print("\n=== Overall distribution of coupling metrics across (Populations, Sample_ID) groups ===")
for metric in ['delta_median_cells', 'effect_size_rank_biserial', 'delta_iqr_cells']:
    if metric in coupling_summary.columns:
        desc = coupling_summary[metric].describe()
        print(f"\nSummary for {metric}:")
        print(desc.to_string())

# Identify globally strongest and weakest groups by delta_median_cells
if 'delta_median_cells' in coupling_summary.columns:
    top_delta = coupling_summary.sort_values('delta_median_cells', ascending=False).head(10)
    bottom_delta = coupling_summary.sort_values('delta_median_cells', ascending=True).head(10)

    print("\nTop 10 (Populations, Sample_ID) groups by strongest median coupling (delta_median_cells):")
    print(top_delta[['Populations', 'Sample_ID', 'delta_median_cells', 'delta_iqr_cells',
                     'effect_size_rank_biserial', 'wilcoxon_fdr']].to_string(index=False))

    print("\nBottom 10 (Populations, Sample_ID) groups by weakest median coupling (delta_median_cells):")
    print(bottom_delta[['Populations', 'Sample_ID', 'delta_median_cells', 'delta_iqr_cells',
                        'effect_size_rank_biserial', 'wilcoxon_fdr']].to_string(index=False))

# Within-sample ranking of populations by coupling strength
print("\n=== Within-sample ranking of populations by coupling strength (delta_median_cells) ===")
if 'delta_median_cells' in coupling_summary.columns:
    for sample_id, sub in coupling_summary.groupby('Sample_ID'):
        sub_sorted = sub.sort_values('delta_median_cells', ascending=False)
        print(f"\nSample_ID: {sample_id}")
        print("Top 5 populations by delta_median_cells:")
        print(sub_sorted.head(5)[['Populations', 'delta_median_cells', 'delta_iqr_cells',
                                  'effect_size_rank_biserial', 'wilcoxon_fdr']].to_string(index=False))
        print("Bottom 5 populations by delta_median_cells:")
        print(sub_sorted.tail(5)[['Populations', 'delta_median_cells', 'delta_iqr_cells',
                                  'effect_size_rank_biserial', 'wilcoxon_fdr']].to_string(index=False))

# Within-population variation in coupling across samples
print("\n=== Within-population variation in coupling across samples ===")
if 'delta_median_cells' in coupling_summary.columns:
    pop_spread_records = []
    for pop, sub in coupling_summary.groupby('Populations'):
        vals = sub['delta_median_cells'].dropna().values
        if vals.size == 0:
            continue
        pop_spread_records.append({
            'Populations': pop,
            'delta_median_min': float(vals.min()),
            'delta_median_max': float(vals.max()),
            'delta_median_range': float(vals.max() - vals.min()),
            'n_samples': sub['Sample_ID'].nunique()
        })
    if pop_spread_records:
        pop_spread_df = pd.DataFrame(pop_spread_records)
        print("\nPopulations with largest across-sample variation in delta_median_cells (top 10 by range):")
        print(pop_spread_df.sort_values('delta_median_range', ascending=False)
              .head(10)[['Populations', 'n_samples', 'delta_median_min', 'delta_median_max', 'delta_median_range']]
              .to_string(index=False))

# Coarse classification of groups by strength and heterogeneity of coupling
print("\n=== Coarse classification of groups by strength and heterogeneity of coupling ===")
if 'delta_median_cells' in coupling_summary.columns and 'delta_iqr_cells' in coupling_summary.columns:
    med_delta_thresh = coupling_summary['delta_median_cells'].median()
    iqr_thresh = coupling_summary['delta_iqr_cells'].median()

    def classify_row(r):
        if not np.isfinite(r['delta_median_cells']) or not np.isfinite(r['delta_iqr_cells']):
            return 'undefined'
        strength = 'strong' if r['delta_median_cells'] >= med_delta_thresh else 'weak'
        het = 'heterogeneous' if r['delta_iqr_cells'] >= iqr_thresh else 'homogeneous'
        return f"{strength}_{het}"

    coupling_summary['strength_heterogeneity_class'] = coupling_summary.apply(classify_row, axis=1)

    class_counts = coupling_summary['strength_heterogeneity_class'].value_counts(dropna=False)
    print("Class counts (groups):")
    print(class_counts.to_string())

    for cls in ['strong_homogeneous', 'strong_heterogeneous', 'weak_homogeneous', 'weak_heterogeneous']:
        sub = coupling_summary[coupling_summary['strength_heterogeneity_class'] == cls]
        if sub.empty:
            continue
        print(f"\nExample groups from class '{cls}' (up to 5):")
        print(sub.sort_values('delta_median_cells', ascending=("weak" in cls))
              .head(5)[['Populations', 'Sample_ID', 'delta_median_cells', 'delta_iqr_cells',
                        'effect_size_rank_biserial', 'wilcoxon_fdr']]
              .to_string(index=False))

# Persist the updated coupling_summary (with heterogeneity and class labels) back to adata.uns
adata.uns['analysis_metadata']['coupling_summary'] = coupling_summary

# ------------------------------------------------------------------
# 3. Prepare merged table with QC for use in the next (correlation) step
# ------------------------------------------------------------------

# qc_pop_sample is assumed to be a pandas DataFrame with a MultiIndex (Populations, Sample_ID)
if isinstance(qc_pop_sample.index, pd.MultiIndex):
    qc_ps = qc_pop_sample.copy().reset_index()
    qc_ps.columns = ['Populations', 'Sample_ID'] + [
        '_'.join([str(c) for c in col]).strip('_') if isinstance(col, tuple) else str(col)
        for col in qc_ps.columns[2:]
    ]
else:
    qc_ps = qc_pop_sample.copy()
    if 'Populations' not in qc_ps.columns or 'Sample_ID' not in qc_ps.columns:
        raise ValueError("qc_summary_pop_sample must have Populations and Sample_ID as index or columns.")

# Merge to create a single table for QC–coupling correlations
merged_for_qc = coupling_summary.merge(
    qc_ps,
    on=['Populations', 'Sample_ID'],
    how='left'
)

adata.uns['analysis_metadata']['coupling_qc_merged'] = merged_for_qc

print("\nUpdated coupling_summary with per-group heterogeneity metrics and coarse strength/heterogeneity classes.")
print("Stored merged coupling–QC table in adata.uns['analysis_metadata']['coupling_qc_merged'] for the next step.")

heterogeneity will only be computed for groups present in coupling_summary.

=== Overall distribution of coupling metrics across (Populations, Sample_ID) groups ===

Summary for delta_median_cells:
count    78.000000
mean      1.912588
std       0.637749
min       0.779816
25%       1.444332
50%       1.876182
75%       2.278498
max       3.515162

Summary for effect_size_rank_biserial:
count    78.000000
mean      0.900988
std       0.075285
min       0.646705
25%       0.865559
50%       0.918117
75%       0.956547
max       0.993415

Summary for delta_iqr_cells:
count    78.000000
mean      1.961411
std       0.368346
min       1.339181
25%       1.704265
50%       1.921558
75%       2.217984
max       2.945670

Top 10 (Populations, Sample_ID) groups by strongest median coupling (delta_median_cells):
Populations Sample_ID  delta_median_cells  delta_iqr_cells  effect_size_rank_biserial  wilcoxon_fdr
         PR   R77_4C4            3.515162         2.485035                   0.991725

### Agent Interpretation

The current step supports the core hypothesis well and sets you up nicely for the QC–coupling correlation step, with a few opportunities to extract more structure from what you already computed.

Key takeaways relative to the hypothesis
----------------------------------------
1. **Strong global spatial–transcriptional coupling.**  
   - All 78 (Population, Sample_ID) groups have **positive delta_median_cells** (min ~0.78, max ~3.5, median ~1.88).  
   - Rank-biserial effect sizes are very high (mean ~0.90, min ~0.65), and all Wilcoxon FDRs are essentially zero.  
   ⇒ Across the board, spatial neighbors are more transcriptomically similar than random same-population cells. This strongly supports the *existence* of spatial–transcriptional coupling.

2. **Substantial variation in coupling strength across groups.**  
   - Top groups (e.g., PR–R77_4C4, PR–R78_4C12, PZ–R78_4C15, PI–R77_4C4, PT–R78_4C15, PK/PM/PF in several samples) have delta_median_cells ~3.0–3.5.  
   - Weakest groups (e.g., PJ, PA, PL, PX, PP, PE, PV) cluster around ~0.8–1.2.  
   - Within each sample, there is a clear stratification of populations into very strong vs relatively weak coupling; this directly satisfies the “differ systematically across (Populations, Sample_ID)” part of the hypothesis.

3. **Non-trivial heterogeneity of coupling within groups.**  
   - delta_iqr_cells spans ~1.34–2.95 with mean ~1.96 and median ~1.92.  
   - Many of the strongest groups (e.g., PR–R77_4C4, PI–R77_4C4, PZ–R78_4C15) also have **high heterogeneity** (delta_iqr_cells > 2.4).  
   - Others (e.g., PN–R78_4C12, PG–R78_4C12, PR–R78_4C15) are “strong_homogeneous”.  
   ⇒ The heterogeneity component of the hypothesis is supported: even within a fixed (Population, Sample_ID), some groups show tight, uniform coupling while others have bimodal/gradient-like or mixed coupling.

4. **Structured variation across samples within the same population.**  
   - Populations like **PR, PI, PX, PZ, PQ, PM, PY, PS, PF, PD** show large ranges of delta_median_cells across samples (e.g., PR range ~1.25, PI ~1.00, PX ~1.00, PZ ~0.96).  
   ⇒ For these populations, spatial–transcriptional coupling is clearly context-dependent across sections, which is exactly what you want to characterize in subsequent steps.

5. **Coarse strength–heterogeneity classification is informative.**  
   - Class counts: 30 strong_heterogeneous, 9 strong_homogeneous, 30 weak_homogeneous, 9 weak_heterogeneous.  
   - The spectrum from **strong_heterogeneous** (PR–R77_4C4, PR–R78_4C12, PZ–R78_4C15, PI–R77_4C4, PT–R78_4C15) to **weak_homogeneous** (PJ, PA, PL across samples) already flags candidate populations for spatial gradients vs uniformly integrated vs uniformly decoupled states.

Concrete suggestions for next steps
-----------------------------------
These build on what you’ve computed while keeping the analysis distinct from typical papers and earlier attempts.

### 1. Leverage the strength–heterogeneity classes more explicitly

You already classified groups into strong/weak × homogeneous/heterogeneous. Before or alongside QC correlations, it would be very informative to:

- **Tabulate per-population class composition across samples.** For each population:
  - Count how many samples are strong_homogeneous, strong_heterogeneous, weak_homogeneous, weak_heterogeneous.
  - This will distinguish populations that are **robustly strongly coupled** (e.g., always strong_homogeneous/strong_heterogeneous) from those that **flip state across samples** (e.g., PR or PI might shift between strong_heterogeneous vs strong_homogeneous vs weaker).

- **Identify “class-switcher” populations.**  
  Focus on populations where:
  - at least one sample is strong_heterogeneous and another is weak_homogeneous or weak_heterogeneous.  
  These are prime candidates for **context-dependent spatial organization** (e.g., anatomical region or developmental stage effects hidden behind Sample_ID).

This requires minimal new code (just groupby over Populations and strength_heterogeneity_class) and gives a high-level, hypothesis-relevant summary.

### 2. In the QC–coupling correlation step, separate strength vs heterogeneity

When you correlate with qc_summary_pop_sample:

- Treat these as **distinct outcomes**:
  - Coupling strength: delta_median_cells, effect_size_rank_biserial.
  - Coupling heterogeneity: delta_iqr_cells (and optionally delta_sd_cells).
- Run **Spearman correlations** for each QC metric against:
  - delta_median_cells,
  - delta_iqr_cells,
  - and (if you want a compact index) +/- classify groups as strong vs weak (binary), heterogeneous vs homogeneous (binary) using the median thresholds you already computed.

Interpretive focus:

- If **QC covaries with delta_median_cells** (e.g., low UMI or complexity → weaker coupling), emphasize that some apparent differences in coupling might be partially technical and worth adjusting for or stratifying by.
- If **QC covaries mainly with delta_iqr_cells** (e.g., low complexity → artificially high heterogeneity, due to noisier distances), that would caution against over-interpreting “heterogeneous” groups without QC matching.
- If correlations are weak or inconsistent, that strengthens the argument that **biological** (not technical) factors drive the observed variation in coupling.

Consider:

- **Stratifying or repeating correlations** separately for strong vs weak groups, or for each Sample_ID, to see whether QC effects are sample-specific vs global.
- Reporting **n_delta_cells** involvement: e.g., check whether groups with very few contributing cells disproportionally show extreme heterogeneity (delta_iqr_cells) or outlier strength; this can be flagged and perhaps down-weighted.

### 3. Exploit populations with large across-sample variation

Your “within-population variation” summary already identified top 10 populations with largest delta_median_cells range (PR, PI, PX, PZ, etc.).

For these, it would be useful to:

- **Overlay their coupling metrics with QC per sample**:
  - For each such population, create a small table: Sample_ID, delta_median_cells, delta_iqr_cells, median UMI, median purity, median complexity, n_delta_cells.
  - This will help you quickly see whether their sample-to-sample coupling variation could be explained by sample-specific QC or is independent of QC.

- In future spatial/gene-level work (not yet coded), these “variable” populations can become targets to:
  - Visualize spatial patterns of high vs low per-cell delta within a sample (e.g., to see whether they map to anatomical subregions).
  - Look for genes whose expression tracks per-cell delta (within a given population–sample).

### 4. Check the “weak but still strongly significant” groups

Some “weak” groups (e.g., PJ, PA, PL, PX, etc.) have:

- Lower delta_median_cells (0.8–1.2), but
- Still quite high effect_size_rank_biserial (0.65–0.83) and ultra-significant Wilcoxon FDR.

This raises interesting biological and technical possibilities:

- They might be:
  - More **spatially intermingled** cell states (e.g., less regionally compartmentalized) that nonetheless retain some local structure.
  - Or they sit in **transitional / boundary regions** where physical proximity does not predict expression as strongly.
- But they also might coincide with:
  - Lower UMI counts or complexity, which you’ll be able to confirm with the merged QC table.

I recommend flagging these weak_homogeneous vs weak_heterogeneous groups explicitly in the QC step, to see whether their weakness is QC-driven or characteristic of particular spatial niches.

### 5. Minor code/robustness suggestions

- You already warn when adata.obs has groups missing from coupling_summary. After the QC merge, it might be worth printing:
  - fraction of (Populations, Sample_ID) with **non-null QC data**, and fraction with **n_delta_cells == 0**, since such groups should be excluded from correlation analyses or at least flagged.
- When computing Spearman correlations later, ensure you:
  - Drop rows with missing values in either the coupling metric or QC metric.
  - Optionally restrict to groups with **n_delta_cells above a small threshold** (e.g., >30) to avoid unstable variance estimates.

Summary of hypothesis status
----------------------------
- The current results **strongly validate** that spatial neighbors are more transcriptomically similar than random same-population cells for nearly all cardiac populations and samples.
- There is **clear, quantifiable variation** in coupling strength (delta_median_cells, effect_size_rank_biserial) and heterogeneity (delta_iqr_cells) across (Populations, Sample_ID) groups and across samples within a population.
- The introduced heterogeneity metrics and coarse strength–heterogeneity classes provide a good, interpretable structure that you can now relate to QC attributes and later to spatial/gene-level features.

The upcoming QC–coupling correlation step is well-posed; the main opportunity is to:
- treat strength vs heterogeneity separately,
- carefully account for n_delta_cells,
- and explicitly examine whether the most biologically intriguing patterns (e.g., PR, PI, PZ, PX, PJ, PA, PL) persist after considering QC.

## Next Steps
Step 1: Using the prepared adata.uns['analysis_metadata']['coupling_qc_merged'] table, quantify how group-level coupling strength (delta_median_cells, effect_size_rank_biserial) and heterogeneity (delta_iqr_cells) covary with QC attributes (median UMI Count, median Purity, median Complexity) across (Populations, Sample_ID) groups via Spearman correlations and simple stratified summaries (e.g., by strength_heterogeneity_class), and interpret whether QC can reasonably explain the observed variability in coupling.

## Compute Spearman correlations between group-level spatial–transcriptional coupling metrics (strength and heterogeneity) and QC attributes across (Populations, Sample_ID) groups, after filtering to groups with sufficient cell support, and provide optional QC summaries stratified by coupling strength–heterogeneity class.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Retrieve merged coupling–QC table
if 'analysis_metadata' not in adata.uns:
    raise ValueError("Missing 'analysis_metadata' in adata.uns; run prior steps first.")

analysis_md = adata.uns['analysis_metadata']

if 'coupling_qc_merged' not in analysis_md:
    raise ValueError("Missing 'coupling_qc_merged' in adata.uns['analysis_metadata']; run the heterogeneity/ranking step first.")

merged = analysis_md['coupling_qc_merged'].copy()

# Inspect available columns
print("Columns in coupling_qc_merged:")
print(sorted(merged.columns))

# Identify key coupling metrics
required_coupling_cols = ['delta_median_cells', 'effect_size_rank_biserial', 'delta_iqr_cells', 'n_delta_cells']
missing_coupling = [c for c in required_coupling_cols if c not in merged.columns]
if missing_coupling:
    raise ValueError(f"Missing required coupling metrics in merged table: {missing_coupling}")

# Heuristic: keep only groups with reasonable per-cell support
min_n_delta = 50
valid_n = merged['n_delta_cells'] >= min_n_delta
print(f"Number of (Populations, Sample_ID) groups with n_delta_cells >= {min_n_delta}:", int(valid_n.sum()))

# Try to infer QC column names (depends on groupby aggregation names)
# We'll look for common patterns in column names
qc_candidates = [c for c in merged.columns if ('UMI' in c or 'Purity' in c or 'Complexity' in c) and ('median' in c or 'mean' in c)]
print("Candidate QC columns (UMI/Purity/Complexity medians/means):", qc_candidates)

# Simple heuristic mapping: prefer median-like columns if present
umi_cols = [c for c in qc_candidates if 'UMI' in c]
purity_cols = [c for c in qc_candidates if 'Purity' in c]
complexity_cols = [c for c in qc_candidates if 'Complexity' in c]

# Choose one representative column for each metric, preferring 'median' over 'mean'
def choose_qc_col(cols, label):
    if not cols:
        print(f"Warning: no QC columns found for {label}; related correlations will be skipped.")
        return None
    med_like = [c for c in cols if 'median' in c.lower()]
    if med_like:
        # if multiple, just take the first alphabetically for reproducibility
        chosen = sorted(med_like)[0]
    else:
        chosen = sorted(cols)[0]
    print(f"Using QC column for {label}: {chosen}")
    return chosen

umi_col = choose_qc_col(umi_cols, 'UMI Count')
purity_col = choose_qc_col(purity_cols, 'Purity')
complexity_col = choose_qc_col(complexity_cols, 'Complexity')

qc_cols_actual = [c for c in [umi_col, purity_col, complexity_col] if c is not None]

# Explicitly log which QC types are present / missing
qc_presence = {
    'UMI_Count_col': umi_col,
    'Purity_col': purity_col,
    'Complexity_col': complexity_col
}
print("QC column selection summary (None = missing):")
for k, v in qc_presence.items():
    print(f"  {k}: {v}")

if not qc_cols_actual:
    raise ValueError("No usable QC columns (UMI/Purity/Complexity) were identified; cannot perform QC–coupling correlation analysis.")

# Helper: compute Spearman correlations between a coupling metric and each QC metric

def spearman_against_qc(df, coupling_col, qc_cols, label, min_n=min_n_delta):
    print(f"\n=== Spearman correlations: {label} vs QC metrics (restricted to n_delta_cells >= {min_n}) ===")
    sub = df[df['n_delta_cells'] >= min_n].copy()
    if sub.empty:
        print("No groups meet the n_delta_cells threshold; skipping.")
        return
    print(f"Total groups used for {label} correlations: {sub.shape[0]}")
    for qc_col in qc_cols:
        # Drop rows with missing values in either variable
        x = sub[coupling_col].astype(float)
        y = sub[qc_col].astype(float)
        mask = np.isfinite(x) & np.isfinite(y)
        n_pairs = int(mask.sum())
        if n_pairs < 10:
            print(f"{label} vs {qc_col}: fewer than 10 valid pairs (n={n_pairs}); skipping.")
            continue
        rho, pval = stats.spearmanr(x[mask], y[mask])
        print(f"{label} vs {qc_col}: n={n_pairs}, Spearman rho={rho:.3f}, p={pval:.3e}")

# 1) Correlations of coupling strength metrics with QC
spearman_against_qc(merged, 'delta_median_cells', qc_cols_actual, 'delta_median_cells')

spearman_against_qc(merged, 'effect_size_rank_biserial', qc_cols_actual, 'effect_size_rank_biserial')

# 2) Correlations of coupling heterogeneity with QC
spearman_against_qc(merged, 'delta_iqr_cells', qc_cols_actual, 'delta_iqr_cells')

# 3) Optional: stratify by coarse strength/heterogeneity class if available
if 'strength_heterogeneity_class' in merged.columns:
    print(f"\n=== QC summaries by strength_heterogeneity_class (restricted to n_delta_cells >= {min_n_delta}) ===")
    sub = merged[merged['n_delta_cells'] >= min_n_delta].copy()
    class_counts = sub['strength_heterogeneity_class'].value_counts(dropna=False)
    print("Class counts (after n_delta_cells filter):")
    print(class_counts.to_string())
    for cls, cls_df in sub.groupby('strength_heterogeneity_class'):
        print(f"\nClass: {cls} (n_groups={cls_df.shape[0]})")
        for qc_col in qc_cols_actual:
            vals = cls_df[qc_col].astype(float)
            valid_vals = vals[np.isfinite(vals)]
            if valid_vals.size == 0:
                print(f"  {qc_col}: no valid values.")
                continue
            desc = valid_vals.describe()
            print(f"  {qc_col} summary:")
            print("   ", desc.to_string().replace("\n", "\n    "))
else:
    print("\nColumn 'strength_heterogeneity_class' not found; skipping class-stratified QC summaries.")

print("\nFinished QC–coupling correlation analysis.")


Columns in coupling_qc_merged:
['Complexity_mean', 'Complexity_median', 'Populations', 'Purity_mean', 'Purity_median', 'Sample_ID', 'UMI Count_mean', 'UMI Count_median', 'd_random_mean_median', 'd_spatial_mean_median', 'delta_iqr', 'delta_iqr_cells', 'delta_mean', 'delta_mean_cells', 'delta_median', 'delta_median_cells', 'delta_sd_cells', 'effect_size_rank_biserial', 'n_cells', 'n_delta_cells', 'n_paired_cells', 'n_valid_cells', 'strength_heterogeneity_class', 'wilcoxon_fdr', 'wilcoxon_pvalue', 'wilcoxon_stat', 'z_score_iqr', 'z_score_mean', 'z_score_median']
Number of (Populations, Sample_ID) groups with n_delta_cells >= 50: 78
Candidate QC columns (UMI/Purity/Complexity medians/means): ['UMI Count_median', 'UMI Count_mean', 'Purity_median', 'Purity_mean', 'Complexity_median', 'Complexity_mean']
Using QC column for UMI Count: UMI Count_median
Using QC column for Purity: Purity_median
Using QC column for Complexity: Complexity_median
QC column selection summary (None = missing):
  UMI_

### Agent Interpretation

These results support the hypothesis that spatial–transcriptional coupling is not just a QC artifact, but they also show that QC covaries with coupling to a non‑trivial extent. I’d summarize the current picture and then suggest how to sharpen it in downstream steps.

**1. What the correlations say about QC vs coupling**

Across 78 (Population, Sample_ID) groups with adequate support:

- **Coupling strength (delta_median_cells)**  
  - vs UMI Count_median: ρ = −0.386, p ≈ 4.8e‑4  
  - vs Purity_median: ρ = +0.371, p ≈ 8.3e‑4  
  - vs Complexity_median: ρ = −0.161, p ≈ 0.16  

  So strength is:
  - *moderately* higher in **lower‑UMI** groups, and
  - *moderately* higher in **higher‑purity** groups,
  - with no clear association to complexity.

- **Alternative strength metric (effect_size_rank_biserial)**  
  - vs UMI Count_median: ρ = −0.320, p ≈ 4.3e‑3  
  - vs Purity_median: ρ = +0.356, p ≈ 1.4e‑3  
  - vs Complexity_median: ρ ≈ 0, p ≈ 0.96  

  Very similar pattern, which increases confidence this is real, not just a quirk of delta_median_cells.

- **Heterogeneity (delta_iqr_cells)**  
  - vs UMI Count_median: ρ = −0.346, p ≈ 1.9e‑3  
  - vs Purity_median: ρ = +0.174, p ≈ 0.13  
  - vs Complexity_median: ρ = −0.215, p ≈ 0.059  

  Heterogeneity also tends to be **higher in lower‑UMI** groups; association with purity is weaker / non‑significant at this n.

Overall, QC explains some variation: ρ ≈ 0.3–0.4 is not negligible, but it’s far from deterministic (R² ~ 0.1–0.15 if linearized). There is plenty of residual variation in coupling that QC does not capture.

The directions also don’t align with a simple “bad QC inflates coupling” story:

- Higher **purity** → stronger coupling (consistent with better cell definition making spatial–expression alignment clearer).
- Lower **UMI depth** → stronger/heterogeneous coupling, which is less intuitive as pure artifact and may reflect genuine differences in transcriptional activity or panel informativeness across regions/populations.

**Conclusion at this stage:** QC is a contributor but not a sufficient explanation for coupling strength/heterogeneity. This is consistent with the hypothesis that there are genuine, group‑specific organizational modes.

---

**2. Class‑stratified QC patterns**

Comparing strength_heterogeneity_class:

- **Weak_homogeneous (n=30)**  
  - Highest UMI: mean ~416; relatively low purity: mean ~0.455; highest complexity: mean ~10.7.
- **Strong_heterogeneous (n=30)**  
  - Lower UMI: mean ~324; higher purity: mean ~0.526; intermediate complexity: mean ~9.3.
- **Strong_homogeneous (n=9)**  
  - UMI similar to strong_het: mean ~325; highest purity: mean ~0.579; high complexity: mean ~10.1.
- **Weak_heterogeneous (n=9)**  
  - UMI similar (~333); intermediate purity (~0.504); intermediate complexity (~9.1).

Key points:

- The **strong** classes do not occupy extreme QC corners. Strong_heterogeneous and strong_homogeneous overlap quite a bit in UMI and complexity, but differ biologically by definition in heterogeneity.
- **Weak_homogeneous** groups actually have *better* UMI and complexity than strong_heterogeneous, but lower purity. So “good UMI/complexity” is not sufficient for strong coupling; purity might be more relevant, but even then doesn’t partition classes cleanly.
- The mix of QC profiles across the four classes indicates that class membership cannot be reduced to “low vs high QC.”

This again aligns with QC as a partial, not dominant, driver.

---

**3. How this informs and refines the hypothesis**

Your hypothesis is *not* “no QC effect,” but that differences in coupling “are not solely driven” by QC. The current data are consistent with that:

- There is a **systematic QC signal** (ρ ~ 0.3–0.4), especially linking purity and lower UMI with stronger coupling.
- Yet, **class structure and residuals** strongly suggest **additional factors**, plausibly biological (cell state, anatomical region, local niche) and/or design‑related (panel match to that cell type).

So I would not over‑correct these QC associations away; instead, treat them as covariates to control and interpret, then ask whether population/sample structure explains residual variation.

---

**4. Concrete next analysis steps to move beyond QC**

To keep things distinct from the paper and prior analyses while building on these results:

1. **QC‑adjusted coupling metrics per group**

   - Fit simple models at the (Population, Sample_ID) level:
     - e.g. `delta_median_cells ~ UMI_median + Purity_median + Complexity_median`
     - and separately `delta_iqr_cells ~ UMI_median + Purity_median + Complexity_median`.
   - Extract **residuals** as “QC‑adjusted coupling strength/heterogeneity.”
   - Examine:
     - Distribution of residuals across **Populations** and **Sample_IDs**.
     - ANOVA or non‑parametric tests: does Population explain residual variation more than Sample_ID, or vice versa?
   - This directly addresses: after accounting for QC, which groups are unusually strongly/heterogeneously coupled?

2. **Within‑QC‑matched comparisons**

   - For each population, identify pairs of Sample_IDs with **similar QC** (e.g., within quantiles / small Euclidean distance in [UMI, purity, complexity] space).
   - Compare coupling metrics between these **QC‑matched** pairs to see if samples still differ substantially.
   - Conversely, within a single Sample_ID, compare populations with similar QC but different coupling.

3. **QC‑stratified spatial patterns**

   - Within high‑purity vs low‑purity tertiles, or high‑UMI vs low‑UMI tertiles:
     - Compare the **spatial patterns** (e.g. spatial autocorrelation of coupling at cell level, or neighborhood composition) of strongly vs weakly coupled groups.
   - The point is to show that even among QC‑matched strata, the **organization mode** (which neighbors they sit next to, how sharply they form domains) differs in a way that pairs with coupling strength/heterogeneity.

4. **Population–sample interaction analysis**

   - Build a simple linear mixed model or two‑way ANOVA on QC‑adjusted coupling:
     - Factors: Population, Sample_ID, and Population×Sample_ID interaction.
   - If the interaction explains significant variance, that’s very consistent with **context‑specific organizational modes** for the same labeled population in different hearts/regions.

5. **Link coupling to spatial neighborhood compositions (not just QC)**

   - For each (Population, Sample_ID) group:
     - quantify **neighbor composition entropy** or specific neighbor enrichments.
   - Correlate these **neighborhood metrics** with QC‑adjusted coupling strength and heterogeneity, instead of with raw QC.
   - This would move the story toward: “After controlling for QC, strong/heterogeneous coupling aligns with particular microenvironmental compositions or boundary structures.”

6. **Sensitivity analyses on QC thresholds**

   - Repeat the correlation and classification summaries with:
     - n_delta_cells thresholds of 30, 100, etc.
     - Alternative QC metrics (median vs mean, or including additional QC fields if present).
   - Show that your key qualitative findings (moderate QC correlation, substantial residual variation) are robust.

---

**5. Interpretation guidance going forward**

- Emphasize that **purity and UMI depth do partially covary with coupling**, so future figures and models should **adjust or stratify by these** to avoid over‑claiming.
- Focus biologically on:
  - Which populations consistently show high QC‑adjusted coupling across samples.
  - Which show **sample‑specific** deviations (high in some hearts, low in others) even at comparable QC.
- Use these QC‑adjusted differences as an entry point to examine **spatial domains, layer structure, or specific neighbor relationships** that might underlie them.

In short, your current step supports the central claim: QC does not fully explain coupling variability. The next steps should explicitly factor QC into models so you can cleanly attribute the remaining structure to population‑ and context‑specific tissue organization.